<div style="text-align:center; padding:35px 0 20px 0; background:linear-gradient(135deg,#0f3460 0%,#16213e 100%); border-radius:12px; margin-bottom:10px;">
<h1 style="color:#e94560; font-size:2.2em; margin:0;">🎓 Learning Behavior Clustering & Student Profiling</h1>
<h2 style="color:#a8dadc; font-size:1.3em; margin:10px 0 0 0; font-weight:400;">End-to-End Machine Learning Pipeline</h2>
<p style="color:#aaa; margin:8px 0 0 0; font-size:0.95em;">ENSIA · Machine Learning Project · Spring 2025–2026</p>
</div>

---
## Project Overview

This notebook presents the complete end-to-end machine learning pipeline for **Learning Behavior Clustering and Student Profiling**. Using anonymized academic and behavioral records, the system applies unsupervised learning techniques to discover meaningful student groups — such as *High Performers*, *Steady Learners*, and *At-Risk Students* — that can inform targeted academic support and evidence-based teaching strategies.

**Pipeline stages covered in this notebook:**

| # | Phase | Description |
|---|-------|-------------|
| 1 | **Data Collection & Preprocessing** | Loading, cleaning, outlier handling, normalization |
| 2 | **Exploratory Data Analysis (EDA)** | Distributions, correlations, feature engineering |
| 3 | **K-Means Clustering** | Optimal k selection, model training, cluster profiles |
| 4 | **DBSCAN Clustering** | Density-based clustering, hyperparameter tuning, outlier detection |
| 5 | **Hierarchical Clustering** | Agglomerative clustering, dendrogram analysis |
| 6 | **Evaluation & Comparison** | Cross-algorithm metrics, visualizations, academic interpretation |

---

## Phase 1 — Data Collection & Preprocessing

### 1.1 Dataset Description

The dataset contains **anonymized academic and behavioral records** of university students. Each row represents one student observation. All personally identifiable information has been removed.

| Property | Value |
|----------|-------|
| **Source** | Anonymized institutional academic records (merged from multiple systems) |
| **Format** | CSV (comma-separated values) |
| **Total Rows (raw)** | 14,003 |
| **Total Features** | 16 |

**Feature groups:** Academic performance (`ExamScore`, `AssignmentCompletion`, `FinalGrade`), Study habits (`StudyHours`, `Attendance`, `OnlineCourses`, `Discussions`, `Extracurricular`), Psychological (`Motivation`, `StressLevel`), Demographic (`Gender`, `Age`, `LearningStyle`), Resources (`Internet`, `Resources`, `EduTech`).

### 1.2 Privacy & Ethics

All data is **fully anonymized** — no names, IDs, or personally identifiable information are present. Data collection complies with institutional data governance policies and is used exclusively for academic research within this project.

### 1.3 Environment Setup

In [1]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import warnings
from pathlib import Path
from math import pi
from collections import Counter

# ── Data manipulation ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cophenet
from scipy.spatial.distance import pdist

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    davies_bouldin_score, calinski_harabasz_score,
)

# ── Global settings ───────────────────────────────────────────────────────────
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.facecolor"] = "white"
np.random.seed(42)

print("✅ All libraries imported successfully.")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   sklearn : {__import__('sklearn').__version__}")

ModuleNotFoundError: No module named 'seaborn'

### 1.4 Load Raw Dataset

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
RAW_DATA_PATH       = Path("../Data/raw/merged_dataset.csv")
PROCESSED_DATA_PATH = Path("../Data/processed/cleaned_dataset.csv")
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Load the raw dataset ──────────────────────────────────────────────────────
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"✅ Dataset loaded successfully.")
print(f"   Path  : {RAW_DATA_PATH}")
print(f"   Shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"\n📌 Column Names & Data Types:")
print(df_raw.dtypes.to_string())
print(f"\n📌 First 5 rows (raw):")
df_raw.head()

### 1.5 Data Cleaning

#### Missing Values

Numerical columns → imputed with **median**; categorical columns → imputed with **mode**. Columns with >40% missing are flagged for removal.

In [ ]:
# ── Work on a copy ────────────────────────────────────────────────────────────
df = df_raw.copy()

# ── Missing value audit ───────────────────────────────────────────────────────
missing_count = df.isnull().sum()
missing_pct   = (missing_count / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %"    : missing_pct,
    "Action"       : ["None needed" if p == 0 else "Drop column" if p > 40 else "Impute"
                      for p in missing_pct]
})
print("📌 Missing Value Report:")
display = missing_report[missing_report["Missing Count"] > 0]
print(display.to_string() if len(display) > 0 else "   ✅ No missing values detected.")

CATEGORICAL_COLS = ["Gender","LearningStyle","Internet","EduTech",
                    "Extracurricular","Discussions","Motivation","StressLevel","FinalGrade"]
NUMERICAL_COLS   = [c for c in df.columns if c not in CATEGORICAL_COLS]

# Drop columns exceeding 40% missing threshold
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f"⚠️  Dropped columns with >40% missing: {cols_to_drop}")
else:
    print("✅ No columns exceed the 40% missing threshold.")

# Impute
for col in NUMERICAL_COLS:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
for col in CATEGORICAL_COLS:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f"\n✅ Missing value handling complete. Remaining nulls: {df.isnull().sum().sum()}")

#### Duplicate Removal

In [ ]:
n_before = len(df)
n_dup = df.duplicated().sum()
print(f"📌 Duplicate rows: {n_dup:,} ({n_dup/n_before*100:.2f}%)")
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"✅ Duplicates removed. Before: {n_before:,} → After: {len(df):,} rows")

#### Outlier Detection & Clipping (IQR Method)

In [ ]:
CONTINUOUS_COLS = ["StudyHours","Attendance","AssignmentCompletion","ExamScore","Age","OnlineCourses"]

print("📌 Outlier Report (IQR method):")
for col in CONTINUOUS_COLS:
    if col not in df.columns: continue
    series = pd.to_numeric(df[col], errors="coerce")
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((series < lower) | (series > upper)).sum()
    if n_out > 0:
        print(f"   ⚠️  '{col}': {n_out} outlier(s) → clipped to [{lower:.1f}, {upper:.1f}]")
        df[col] = series.clip(lower, upper)
    else:
        print(f"   ✅ '{col}': no outliers detected")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(CONTINUOUS_COLS):
    ax = axes[i//3, i%3]
    ax.boxplot(pd.to_numeric(df[col], errors="coerce").dropna(), patch_artist=True,
               boxprops=dict(facecolor="#4C72B0", alpha=0.6),
               medianprops=dict(color="#C44E52", linewidth=2))
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.grid(axis="y", alpha=0.4)
plt.suptitle("Boxplots of Numerical Features (after cleaning)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

#### Data Type Corrections & Min-Max Normalization

In [ ]:
INT8_COLS = ["Gender","LearningStyle","Internet","EduTech","Extracurricular",
             "Discussions","Motivation","StressLevel","FinalGrade","Resources"]
for col in INT8_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int8")
for col in CONTINUOUS_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")

# Min-Max normalization on continuous columns
scaler_mm = MinMaxScaler()
df[CONTINUOUS_COLS] = scaler_mm.fit_transform(df[CONTINUOUS_COLS]).round(4)

print(f"✅ Normalization complete. Dataset shape: {df.shape}")
print(df[CONTINUOUS_COLS].describe().round(4))

### 1.6 Save Cleaned Dataset

In [ ]:
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"✅ Cleaned dataset saved → {PROCESSED_DATA_PATH}")
print(f"   Final shape: {df.shape}")

# Summary
summary_table = pd.DataFrame({
    "Task": ["Dataset loaded","Duplicates removed","Outlier handling","Normalization","Saved"],
    "Status": ["✅","✅","✅","✅","✅"],
    "Details": [f"{df_raw.shape[0]:,} rows × {df_raw.shape[1]} cols",
                f"{n_dup:,} rows removed → {len(df):,} clean",
                "IQR method; clipped to bounds",
                "Min-Max [0,1] on 6 continuous cols",
                str(PROCESSED_DATA_PATH)]
})
print("\n📌 Phase 1 Summary:")
print(summary_table.to_string(index=False))

---
## Phase 2 — Exploratory Data Analysis (EDA)

In this phase we explore the cleaned dataset to understand feature distributions, detect remaining anomalies, analyze correlations, and engineer new features that improve clustering quality.

### 2.1 Load Cleaned Dataset & Column Classification

In [ ]:
df_eda = pd.read_csv(PROCESSED_DATA_PATH)
print(f"✅ Cleaned dataset loaded. Shape: {df_eda.shape}")
print(df_eda.dtypes.value_counts())

numeric_cols = df_eda.select_dtypes(include='number').columns.tolist()
binary_cols  = [c for c in df_eda.columns if df_eda[c].nunique(dropna=True) == 2]
ordinal_cols = [c for c in numeric_cols if df_eda[c].nunique(dropna=True) <= 10 and c not in binary_cols]
cont_cols    = [c for c in numeric_cols if c not in ordinal_cols and c not in binary_cols]

print(f"\n📌 Continuous numeric : {cont_cols}")
print(f"   Ordinal / discrete : {ordinal_cols}")
print(f"   Binary             : {binary_cols}")

### 2.2 Descriptive Statistics

In [ ]:
print("📌 Continuous features:")
print(df_eda[cont_cols].describe().T.round(4))
print("\n📌 Ordinal/discrete features:")
print(df_eda[ordinal_cols].describe().T.round(4))

### 2.3 Distribution & Outlier Analysis

In [ ]:
# Histograms for continuous features
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, col in enumerate(cont_cols[:6]):
    ax = axes[i//3, i%3]
    sns.histplot(df_eda[col], bins=25, kde=True, ax=ax, color="#4C72B0")
    ax.set_title(col, fontweight="bold")
plt.suptitle("Distribution of Continuous Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Skewness & Kurtosis
from scipy import stats as sp_stats
skew_kurt = pd.DataFrame({
    'skewness': df_eda[numeric_cols].skew(),
    'kurtosis': df_eda[numeric_cols].kurt()
})
skew_kurt['skew_flag']     = skew_kurt['skewness'].apply(lambda x: '⚠️ check' if abs(x)>1 else '✅ ok')
skew_kurt['kurtosis_flag'] = skew_kurt['kurtosis'].apply(lambda x: 'heavy_tail' if x>3 else 'ok')
print("📌 Skewness & Kurtosis:")
print(skew_kurt.sort_values('skewness', key=lambda s: s.abs(), ascending=False).round(4))

### 2.4 Correlation Analysis

In [ ]:
# Spearman correlation heatmap
corr = df_eda[numeric_cols].corr(method='spearman')
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f',
            linewidths=0.4, linecolor='white', ax=ax)
ax.set_title('Spearman Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations
corr_abs = corr.abs()
upper    = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
print("📌 Top 10 absolute correlations:")
print(upper.stack().sort_values(ascending=False).head(10).round(4))

### 2.5 Feature Engineering

Two derived features are created to capture complementary aspects of student behavior more meaningfully than raw variables alone:

In [ ]:
df_fe = df_eda.copy()

# ── Derived features ──────────────────────────────────────────────────────────
# engagement_score: combines four engagement signals into one composite indicator
df_fe['engagement_score'] = df_fe[['StudyHours','Attendance','AssignmentCompletion','Discussions']].mean(axis=1)

# avg_session_time: study intensity relative to online course load
df_fe['avg_session_time'] = df_fe['StudyHours'] / df_fe['OnlineCourses'].clip(lower=0.01)

print("✅ Derived features added:")
print("   • engagement_score  — composite of StudyHours, Attendance, AssignmentCompletion, Discussions")
print("   • avg_session_time  — StudyHours / OnlineCourses (clipped to avoid division by zero)")
print(f"\n   Updated shape: {df_fe.shape}")
df_fe[['engagement_score','avg_session_time']].describe().round(4)

### 2.6 Scaling & PCA Preview

In [ ]:
MODELING_PATH = Path("../Data/data_modeling.csv")

# Features for clustering (exclude target-like columns)
EXCLUDE = ['FinalGrade', 'ExamScore', 'Gender', 'LearningStyle']
candidate_cols = [c for c in df_fe.select_dtypes(include='number').columns if c not in EXCLUDE]

scaler_std = StandardScaler()
X_eda_scaled = scaler_std.fit_transform(df_fe[candidate_cols])
X_eda_df     = pd.DataFrame(X_eda_scaled, columns=candidate_cols)

# Remove highly correlated features (> 0.90)
corr_m  = X_eda_df.corr().abs()
upper_m = corr_m.where(np.triu(np.ones(corr_m.shape), k=1).astype(bool))
drop_hc = [col for col in upper_m.columns if any(upper_m[col] > 0.90)]
chosen  = [c for c in candidate_cols if c not in drop_hc]
X_model = X_eda_df[chosen].copy()

print(f"📌 High-correlation drop (>0.90): {drop_hc}")
print(f"   Final modeling features ({len(chosen)}): {chosen}")
print(f"   X_model shape: {X_model.shape}")

X_model.to_csv(MODELING_PATH, index=False)
print(f"\n✅ Modeling dataset saved → {MODELING_PATH}")

# PCA preview
pca_prev = PCA(n_components=2, random_state=42)
pca_xy   = pca_prev.fit_transform(X_model)
sample_p = pd.DataFrame(pca_xy, columns=['PC1','PC2']).sample(n=min(2000, len(pca_xy)), random_state=42)
plt.figure(figsize=(7,5))
plt.scatter(sample_p['PC1'], sample_p['PC2'], s=10, alpha=0.5, color='#4C72B0')
plt.title(f"PCA Preview ({pca_prev.explained_variance_ratio_.sum():.2%} variance explained)",
          fontsize=12, fontweight='bold')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.tight_layout()
plt.show()

---
## Phase 3 — K-Means Clustering

K-Means is selected as the baseline algorithm: computationally efficient, scalable to large cohorts, and producing interpretable centroid-based profiles suitable for academic advising.

### 3.1 Load Data & Feature Matrix

In [ ]:
df_km = pd.read_csv(PROCESSED_DATA_PATH)

CLUSTERING_FEATURES_KM = [
    "StudyHours","Attendance","Resources","Extracurricular","Motivation",
    "Internet","Age","OnlineCourses","Discussions","AssignmentCompletion",
    "ExamScore","EduTech","StressLevel","FinalGrade",
]

X_km = df_km[CLUSTERING_FEATURES_KM].copy()
missing_km = X_km.isnull().sum()
if missing_km.sum() > 0:
    X_km.fillna(X_km.median(), inplace=True)

scaler_km = StandardScaler()
X_km_scaled = scaler_km.fit_transform(X_km)
X_km_df     = pd.DataFrame(X_km_scaled, columns=CLUSTERING_FEATURES_KM)

print(f"✅ Feature matrix prepared — shape: {X_km_scaled.shape}")

### 3.2 Optimal k Selection

Three complementary methods are used: **Elbow (WCSS)**, **Silhouette Score**, and **Davies-Bouldin Index**.

In [ ]:
k_range        = range(2, 11)
wcss_list      = []
sil_km_list    = []
dbi_km_list    = []

print("🔄 Computing metrics for k = 2 to 10 ...")
for k in k_range:
    km_ = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    lbl_ = km_.fit_predict(X_km_scaled)
    wcss_list.append(km_.inertia_)
    sil_km_list.append(silhouette_score(X_km_scaled, lbl_))
    dbi_km_list.append(davies_bouldin_score(X_km_scaled, lbl_))
    print(f"   k={k}  │  WCSS={km_.inertia_:.2f}  │  Sil={sil_km_list[-1]:.4f}  │  DBI={dbi_km_list[-1]:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(k_range, wcss_list, marker='o', linewidth=2.5, color='#2E86AB', markerfacecolor='#A23B72')
axes[0].set_title('Elbow Method (WCSS)', fontweight='bold'); axes[0].set_xlabel('k'); axes[0].grid(True, alpha=0.3)

best_sil_k = list(k_range)[np.argmax(sil_km_list)]
axes[1].plot(k_range, sil_km_list, marker='s', linewidth=2.5, color='#06A77D', markerfacecolor='#F18F01')
axes[1].axvline(best_sil_k, color='red', linestyle='--', linewidth=2, label=f'Best k={best_sil_k}')
axes[1].set_title('Silhouette Score', fontweight='bold'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

best_dbi_k = list(k_range)[np.argmin(dbi_km_list)]
axes[2].plot(k_range, dbi_km_list, marker='^', linewidth=2.5, color='#C1121F', markerfacecolor='#FFC300')
axes[2].axvline(best_dbi_k, color='green', linestyle='--', linewidth=2, label=f'Best k={best_dbi_k}')
axes[2].set_title('Davies-Bouldin Index (↓ better)', fontweight='bold'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

for ax in axes: ax.set_xticks(k_range)
plt.suptitle('Optimal k Selection — Three Methods', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print(f"\n📌 Best k by Silhouette: {best_sil_k}  |  Best k by DBI: {best_dbi_k}")

### 3.3 Train Final K-Means Model

All three methods converge on **k = 3**, providing the strongest internal-metric support while preserving practical interpretability.

In [ ]:
OPTIMAL_K_KM = 3

kmeans_final = KMeans(n_clusters=OPTIMAL_K_KM, random_state=42, n_init=10, max_iter=300)
df_km['Cluster'] = kmeans_final.fit_predict(X_km_scaled)

print(f"✅ K-Means trained (k={OPTIMAL_K_KM})  |  Inertia: {kmeans_final.inertia_:.2f}")
counts_km = df_km['Cluster'].value_counts().sort_index()
for cid, cnt in counts_km.items():
    print(f"   Cluster {cid}: {cnt:,} students ({cnt/len(df_km)*100:.1f}%)")

### 3.4 Cluster Evaluation

In [ ]:
sil_km = silhouette_score(X_km_scaled, df_km['Cluster'])
dbi_km = davies_bouldin_score(X_km_scaled, df_km['Cluster'])
ch_km  = calinski_harabasz_score(X_km_scaled, df_km['Cluster'])

print("=" * 60)
print(f"  K-MEANS EVALUATION METRICS (k={OPTIMAL_K_KM})")
print("=" * 60)
print(f"  Silhouette Score       : {sil_km:.4f}  (range [-1,1]; higher better)")
print(f"  Davies-Bouldin Index   : {dbi_km:.4f}  (lower better; <1 = well-separated)")
print(f"  Calinski-Harabasz      : {ch_km:.2f}  (higher better)")
print("=" * 60)

### 3.5 Visualization — PCA & t-SNE

In [ ]:
pca_km = PCA(n_components=2, random_state=42)
X_pca_km = pca_km.fit_transform(X_km_scaled)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# PCA
sc = axes[0].scatter(X_pca_km[:,0], X_pca_km[:,1], c=df_km['Cluster'],
                     cmap='Set2', s=30, alpha=0.6, edgecolors='black', linewidth=0.3)
centers_pca = pca_km.transform(kmeans_final.cluster_centers_)
axes[0].scatter(centers_pca[:,0], centers_pca[:,1], c='red', marker='*', s=500,
                edgecolors='darkred', linewidth=1.5, label='Centers', zorder=5)
axes[0].set_xlabel(f'PC1 ({pca_km.explained_variance_ratio_[0]:.1%} var)', fontweight='bold')
axes[0].set_ylabel(f'PC2 ({pca_km.explained_variance_ratio_[1]:.1%} var)', fontweight='bold')
axes[0].set_title('K-Means — PCA Projection', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.25)

# t-SNE
print("🔄 Running t-SNE (may take a moment)...")
tsne_km = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne_km = tsne_km.fit_transform(X_km_scaled)
axes[1].scatter(X_tsne_km[:,0], X_tsne_km[:,1], c=df_km['Cluster'],
                cmap='Set2', s=30, alpha=0.6, edgecolors='black', linewidth=0.3)
axes[1].set_xlabel('t-SNE dim 1', fontweight='bold')
axes[1].set_ylabel('t-SNE dim 2', fontweight='bold')
axes[1].set_title('K-Means — t-SNE Projection', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.25)

plt.suptitle('K-Means Cluster Visualization', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print("✅ t-SNE complete.")

### 3.6 Cluster Profiling & Interpretation

In [ ]:
centroids_km = scaler_km.inverse_transform(kmeans_final.cluster_centers_)
centroids_df_km = pd.DataFrame(centroids_km, columns=CLUSTERING_FEATURES_KM)

# Auto-label based on ExamScore ranking
ranked_km = centroids_df_km['ExamScore'].sort_values(ascending=False).index.tolist()
km_labels_map = {}
km_labels_map[ranked_km[0]]  = "🟢 High Performers"
km_labels_map[ranked_km[-1]] = "🔴 At-Risk / Struggling Students"
for cid in ranked_km[1:-1]:
    km_labels_map[cid] = "🟡 Average / Steady Learners"

df_km['KM_Profile'] = df_km['Cluster'].map(km_labels_map)

print("=" * 80)
print("  K-MEANS CLUSTER PROFILES")
print("=" * 80)
for cid in ranked_km:
    cnt = (df_km['Cluster'] == cid).sum()
    print(f"\n{km_labels_map[cid]}  (Cluster {cid})")
    print(f"  Size: {cnt:,} students ({cnt/len(df_km)*100:.1f}%)")
    for feat in ["ExamScore","Attendance","StudyHours","AssignmentCompletion","Motivation","StressLevel"]:
        print(f"  • {feat:<25} {centroids_df_km.loc[cid, feat]:.3f}")

# Boxplots for key features
key_feat = ["StudyHours","Attendance","AssignmentCompletion","ExamScore","Motivation","StressLevel"]
fig, axes = plt.subplots(2, 3, figsize=(16,10))
for ax, feat in zip(axes.flatten(), key_feat):
    sns.boxplot(data=df_km, x='Cluster', y=feat, ax=ax, palette='Set2')
    ax.set_title(feat, fontweight='bold')
    ax.grid(True, alpha=0.25, axis='y')
plt.suptitle('K-Means — Feature Distributions by Cluster', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 3.7 At-Risk Student Identification

In [ ]:
at_risk_km_id = centroids_df_km['ExamScore'].idxmin()
at_risk_km = df_km[df_km['Cluster'] == at_risk_km_id]
print(f"📌 At-Risk Cluster: Cluster {at_risk_km_id} ({km_labels_map[at_risk_km_id]})")
print(f"   Students: {len(at_risk_km):,} ({len(at_risk_km)/len(df_km)*100:.1f}%)")
print("\n📊 At-Risk Summary Statistics:")
print(at_risk_km[["ExamScore","Attendance","StudyHours","AssignmentCompletion","Motivation"]].describe().round(3))

### 3.8 Clustering Stability Check

In [ ]:
stability_km = []
print("🔄 Stability check (5 × 80% subsamples)...")
for it in range(5):
    idx_s = np.random.choice(len(X_km_scaled), int(0.8*len(X_km_scaled)), replace=False)
    km_s  = KMeans(n_clusters=OPTIMAL_K_KM, random_state=42, n_init=10)
    lbl_s = km_s.fit_predict(X_km_scaled[idx_s])
    sil_s = silhouette_score(X_km_scaled[idx_s], lbl_s)
    stability_km.append(sil_s)
    print(f"   Iteration {it+1}: Silhouette = {sil_s:.4f}")

mean_stab_km = np.mean(stability_km)
std_stab_km  = np.std(stability_km)
print(f"\n✅ Mean = {mean_stab_km:.4f}  |  Std = {std_stab_km:.4f}")
print(f"   → {'✅ STABLE' if std_stab_km < 0.05 else '⚠️ VARIABLE'}")

df_km_out = Path("../Data/processed/kmeans_clustered.csv")
df_km.to_csv(df_km_out, index=False)
print(f"\n✅ K-Means results saved → {df_km_out}")

---
## Phase 4 — DBSCAN Clustering

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) addresses three key limitations of K-Means: it handles arbitrary cluster shapes, does not require k to be pre-specified, and provides built-in outlier detection.

### 4.1 Load Data & Feature Matrix

In [ ]:
df_dbs = pd.read_csv(PROCESSED_DATA_PATH)

CLUSTERING_FEATURES_DBS = [
    "StudyHours","Attendance","Extracurricular","Motivation","OnlineCourses",
    "Discussions","AssignmentCompletion","ExamScore","StressLevel","FinalGrade",
]

X_dbs = df_dbs[CLUSTERING_FEATURES_DBS].copy()
if X_dbs.isnull().sum().sum() > 0:
    X_dbs.fillna(X_dbs.median(), inplace=True)

scaler_dbs  = StandardScaler()
X_dbs_scaled = scaler_dbs.fit_transform(X_dbs)

print(f"✅ Feature matrix prepared — shape: {X_dbs_scaled.shape}")

### 4.2 Hyperparameter Tuning

**`min_samples`** heuristic: ≥ dimensionality + 1. **`eps`** is estimated via the k-distance graph elbow.

In [ ]:
MIN_SAMPLES_DEFAULT = 15
k_nn = MIN_SAMPLES_DEFAULT - 1

print(f"🔄 Computing {k_nn}-NN distances ...")
nbrs_dbs = NearestNeighbors(n_neighbors=k_nn, algorithm="ball_tree", n_jobs=-1)
nbrs_dbs.fit(X_dbs_scaled)
distances_dbs, _ = nbrs_dbs.kneighbors(X_dbs_scaled)

k_dists = np.sort(distances_dbs[:, -1])[::-1]
x_d = np.arange(len(k_dists))
dx_d = np.gradient(k_dists, x_d); ddx_d = np.gradient(dx_d, x_d)
curv_d = np.abs(ddx_d) / (1 + dx_d**2)**1.5
lo_d, hi_d = int(0.05*len(k_dists)), int(0.60*len(k_dists))
elbow_i = lo_d + np.argmax(curv_d[lo_d:hi_d])
SUGGESTED_EPS = round(float(k_dists[elbow_i]), 3)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_dists, linewidth=1.5, color="#2E86AB", label="k-distance curve")
ax.axvline(elbow_i, color="#e94560", linestyle="--", linewidth=2, label=f"Elbow → eps ≈ {SUGGESTED_EPS}")
ax.axhline(SUGGESTED_EPS, color="orange", linestyle=":", linewidth=1.5, alpha=0.7)
ax.set_xlabel(f"Points sorted by distance (descending)", fontweight="bold")
ax.set_ylabel(f"Distance to {k_nn}-th nearest neighbour", fontweight="bold")
ax.set_title(f"k-Distance Graph  (min_samples={MIN_SAMPLES_DEFAULT})", fontsize=13, fontweight="bold")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"\n📌 Suggested eps: {SUGGESTED_EPS}")

#### Grid Search over (eps, min_samples)

In [ ]:
eps_candidates = np.round(np.arange(max(0.10, SUGGESTED_EPS - 0.40), SUGGESTED_EPS + 0.55, 0.05), 3)
min_samples_candidates = [10, 15, 20, 25, 30]

grid_results = []
total_gs = len(eps_candidates) * len(min_samples_candidates)
print(f"🔄 Grid search: {total_gs} configurations ...")
for eps_v in eps_candidates:
    for ms in min_samples_candidates:
        db_ = DBSCAN(eps=eps_v, min_samples=ms, n_jobs=-1)
        lbl_ = db_.fit_predict(X_dbs_scaled)
        n_cls_ = len(set(lbl_)) - (1 if -1 in lbl_ else 0)
        noise_  = (lbl_ == -1).mean()
        if n_cls_ >= 2 and noise_ < 0.50:
            sil_ = silhouette_score(X_dbs_scaled, lbl_, sample_size=5000, random_state=42)
            dbi_ = davies_bouldin_score(X_dbs_scaled, lbl_)
            chi_ = calinski_harabasz_score(X_dbs_scaled, lbl_)
        else:
            sil_ = dbi_ = chi_ = np.nan
        grid_results.append(dict(eps=eps_v, min_samples=ms, n_clusters=n_cls_,
                                 noise_pct=round(noise_*100,2), silhouette=sil_, dbi=dbi_, ch=chi_))

grid_df = pd.DataFrame(grid_results)
valid_df_dbs = grid_df.dropna(subset=["silhouette"]).copy()
print(f"\n✅ {len(valid_df_dbs)} valid configurations found.")
print("\n📌 Top 10 by Silhouette Score:")
print(valid_df_dbs.sort_values("silhouette", ascending=False).head(10).to_string(index=False))

### 4.3 Train Final DBSCAN Model

In [ ]:
filtered_dbs = valid_df_dbs[(valid_df_dbs["noise_pct"] < 30) & (valid_df_dbs["n_clusters"] >= 2)]
if filtered_dbs.empty: filtered_dbs = valid_df_dbs
best_row_dbs = filtered_dbs.loc[filtered_dbs["silhouette"].idxmax()]
BEST_EPS_DBS = best_row_dbs["eps"]
BEST_MS_DBS  = int(best_row_dbs["min_samples"])

print(f"🏆 Best configuration: eps={BEST_EPS_DBS}, min_samples={BEST_MS_DBS}")
print(f"   Silhouette={best_row_dbs['silhouette']:.4f}  |  DBI={best_row_dbs['dbi']:.4f}")
print(f"   n_clusters={int(best_row_dbs['n_clusters'])}  |  Noise={best_row_dbs['noise_pct']:.2f}%")

dbscan_final = DBSCAN(eps=BEST_EPS_DBS, min_samples=BEST_MS_DBS, n_jobs=-1)
DBSCAN_LABELS = dbscan_final.fit_predict(X_dbs_scaled)
df_dbs["DBSCAN_Cluster"] = DBSCAN_LABELS

N_DBS_CLS  = len(set(DBSCAN_LABELS)) - (1 if -1 in DBSCAN_LABELS else 0)
N_DBS_NOISE = (DBSCAN_LABELS == -1).sum()
DBS_NOISE_PCT = N_DBS_NOISE / len(DBSCAN_LABELS) * 100

print(f"\n✅ DBSCAN trained — {N_DBS_CLS} clusters discovered, {N_DBS_NOISE:,} noise points ({DBS_NOISE_PCT:.2f}%)")
print(pd.Series(DBSCAN_LABELS).value_counts().sort_index()
      .rename(index=lambda i: "Noise (-1)" if i==-1 else f"Cluster {i}").to_string())

### 4.4 Cluster Quality Evaluation

In [ ]:
mask_dbs    = DBSCAN_LABELS != -1
X_dbs_clean = X_dbs_scaled[mask_dbs]
lbl_dbs_cln = DBSCAN_LABELS[mask_dbs]

DBS_SIL = silhouette_score(X_dbs_clean, lbl_dbs_cln, sample_size=5000, random_state=42)
DBS_DBI = davies_bouldin_score(X_dbs_clean, lbl_dbs_cln)
DBS_CHI = calinski_harabasz_score(X_dbs_clean, lbl_dbs_cln)

print("=" * 60)
print("  DBSCAN EVALUATION METRICS  (noise points excluded)")
print("=" * 60)
print(f"  Silhouette Score       : {DBS_SIL:.4f}")
print(f"  Davies-Bouldin Index   : {DBS_DBI:.4f}")
print(f"  Calinski-Harabasz      : {DBS_CHI:.2f}")
print("=" * 60)

### 4.5 Visualization — PCA & t-SNE

In [ ]:
pca_dbs   = PCA(n_components=2, random_state=42)
X_pca_dbs = pca_dbs.fit_transform(X_dbs_scaled)
exp_dbs   = pca_dbs.explained_variance_ratio_ * 100

unique_dbs_lbl = sorted(set(DBSCAN_LABELS))
cmap_dbs = plt.cm.get_cmap("tab10", len(unique_dbs_lbl))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# PCA
for i, lbl in enumerate(unique_dbs_lbl):
    m = DBSCAN_LABELS == lbl
    c = "#bbbbbb" if lbl == -1 else cmap_dbs(i)
    a = 0.25 if lbl == -1 else 0.70
    lab = "Noise (−1)" if lbl == -1 else f"Cluster {lbl}"
    axes[0].scatter(X_pca_dbs[m,0], X_pca_dbs[m,1], c=[c], alpha=a, s=15 if lbl==-1 else 25, label=lab)
axes[0].set_xlabel(f"PC1 ({exp_dbs[0]:.1f}% var)", fontweight="bold")
axes[0].set_ylabel(f"PC2 ({exp_dbs[1]:.1f}% var)", fontweight="bold")
axes[0].set_title("DBSCAN — PCA Projection", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=9, markerscale=2); axes[0].grid(True, alpha=0.2)

# t-SNE
TSNE_N_DBS = min(4000, len(X_dbs_scaled))
idx_t_dbs  = np.random.choice(len(X_dbs_scaled), TSNE_N_DBS, replace=False)
print("🔄 Running t-SNE ...")
tsne_dbs   = TSNE(n_components=2, perplexity=40, learning_rate="auto", init="pca", random_state=42)
X_t_dbs    = tsne_dbs.fit_transform(X_dbs_scaled[idx_t_dbs])
lbl_t_dbs  = DBSCAN_LABELS[idx_t_dbs]
for i, lbl in enumerate(unique_dbs_lbl):
    m = lbl_t_dbs == lbl
    c = "#bbbbbb" if lbl == -1 else cmap_dbs(i)
    lab = "Noise (−1)" if lbl == -1 else f"Cluster {lbl}"
    axes[1].scatter(X_t_dbs[m,0], X_t_dbs[m,1], c=[c], alpha=0.25 if lbl==-1 else 0.75, s=15, label=lab)
axes[1].set_xlabel("t-SNE dim 1", fontweight="bold"); axes[1].set_ylabel("t-SNE dim 2", fontweight="bold")
axes[1].set_title(f"DBSCAN — t-SNE Projection (n={TSNE_N_DBS:,})", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=9, markerscale=2); axes[1].grid(True, alpha=0.2)
print("✅ t-SNE complete.")

plt.suptitle("DBSCAN Cluster Visualization", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

### 4.6 Cluster Profiling & Noise Analysis

In [ ]:
# Feature profiles
profile_dbs = df_dbs.groupby("DBSCAN_Cluster")[CLUSTERING_FEATURES_DBS].mean().T
profile_dbs.columns = [f"Noise (-1)" if c==-1 else f"Cluster {c}" for c in profile_dbs.columns]

fig, ax = plt.subplots(figsize=(max(8, len(profile_dbs.columns)*2+2), 8))
p_norm = (profile_dbs - profile_dbs.min(axis=1).values.reshape(-1,1)) /          (profile_dbs.max(axis=1) - profile_dbs.min(axis=1)).values.reshape(-1,1)
sns.heatmap(p_norm, annot=profile_dbs.round(2), fmt=".2f", cmap="RdYlGn",
            linewidths=0.5, linecolor="white", ax=ax,
            cbar_kws={"label":"Normalised mean (0–1)", "shrink":0.7})
ax.set_title("DBSCAN — Cluster Feature Heatmap", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout(); plt.show()

# Noise vs cluster comparison
noise_mask_dbs = df_dbs["DBSCAN_Cluster"] == -1
comp_dbs = pd.concat([
    df_dbs[noise_mask_dbs][CLUSTERING_FEATURES_DBS].mean().rename("Noise (outliers)"),
    df_dbs[~noise_mask_dbs][CLUSTERING_FEATURES_DBS].mean().rename("Cluster members"),
], axis=1)
comp_dbs["Δ"] = comp_dbs["Noise (outliers)"] - comp_dbs["Cluster members"]
print("📊 Noise vs Cluster Members — Mean Feature Comparison:")
print(comp_dbs.round(4).to_string())

In [ ]:
dbs_out_path = Path("../Data/processed/dbscan_labelled_dataset.csv")
df_dbs.to_csv(dbs_out_path, index=False)
print(f"✅ DBSCAN results saved → {dbs_out_path}")

---
## Phase 5 — Hierarchical Clustering

Hierarchical (Agglomerative) clustering builds a tree of merges (dendrogram) that reveals nested structure at every granularity level — without fixing k in advance.

### 5.1 Load Data & Feature Matrix

In [ ]:
df_hc = pd.read_csv(PROCESSED_DATA_PATH)

CLUSTERING_FEATURES_HC = [
    "StudyHours","Attendance","Extracurricular","Motivation","OnlineCourses",
    "Discussions","AssignmentCompletion","ExamScore","StressLevel","FinalGrade",
]

X_hc = df_hc[CLUSTERING_FEATURES_HC].copy()
if X_hc.isnull().sum().sum() > 0:
    X_hc.fillna(X_hc.median(), inplace=True)

scaler_hc  = StandardScaler()
X_hc_scaled = scaler_hc.fit_transform(X_hc)
X_hc_df     = pd.DataFrame(X_hc_scaled, columns=CLUSTERING_FEATURES_HC, index=df_hc.index)

print(f"✅ Feature matrix prepared — shape: {X_hc_scaled.shape}")

### 5.2 Linkage Method Selection

We compare all four linkage methods using the **Cophenetic Correlation Coefficient (CCC)** — higher is better (> 0.75 is a good fit).

> ⚠️ Scipy linkage has O(n²) memory complexity. We **subsample to 2,000 points** for dendrogram construction, then use `AgglomerativeClustering` on the full dataset.

In [ ]:
DENDRO_SAMPLE = 2000
idx_sub = np.random.choice(len(X_hc_scaled), DENDRO_SAMPLE, replace=False)
X_sub   = X_hc_scaled[idx_sub]

LINKAGE_METHODS_HC = ["ward","complete","average","single"]
linkage_matrices_hc = {}
ccc_scores_hc       = {}

print("🔄 Computing CCC for all linkage methods ...")
for method in LINKAGE_METHODS_HC:
    if method == "ward":
        Z = linkage(X_sub, method=method)
    else:
        Z = linkage(pdist(X_sub, metric="euclidean"), method=method)
    c, _ = cophenet(Z, pdist(X_sub))
    linkage_matrices_hc[method] = Z
    ccc_scores_hc[method]       = c
    print(f"   {method:10s} → CCC = {c:.4f}")

best_linkage_hc = max(ccc_scores_hc, key=ccc_scores_hc.get)
print(f"\n🏆 Best linkage: '{best_linkage_hc}' (CCC = {ccc_scores_hc[best_linkage_hc]:.4f})")

# CCC bar chart
colors_ccc = ["#e94560" if m == best_linkage_hc else "#2E86AB" for m in LINKAGE_METHODS_HC]
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(LINKAGE_METHODS_HC, [ccc_scores_hc[m] for m in LINKAGE_METHODS_HC],
              color=colors_ccc, alpha=0.88, edgecolor="white", linewidth=1.5)
ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=11, fontweight="bold")
ax.axhline(0.75, color="grey", linestyle="--", linewidth=1.5, alpha=0.7, label="CCC = 0.75 threshold")
ax.set_xlabel("Linkage Method", fontweight="bold"); ax.set_ylabel("CCC", fontweight="bold")
ax.set_title("Linkage Method Comparison — Cophenetic Correlation Coefficient",
             fontsize=13, fontweight="bold"); ax.set_ylim(0, 1.05)
ax.legend(); ax.grid(True, alpha=0.25, axis="y")
plt.tight_layout(); plt.show()

### 5.3 Dendrogram Analysis

In [ ]:
# Four-linkage comparison
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
for ax, method in zip(axes.flatten(), LINKAGE_METHODS_HC):
    Z = linkage_matrices_hc[method]
    cut = 0.70 * Z[:, 2].max()
    dendrogram(Z, ax=ax, truncate_mode="lastp", p=50, color_threshold=cut,
               above_threshold_color="#aaaaaa", leaf_rotation=90, leaf_font_size=7, show_contracted=True)
    ax.axhline(y=cut, color="#e94560", linestyle="--", linewidth=1.8)
    ax.set_title(f"Linkage: {method.upper()}  |  CCC = {ccc_scores_hc[method]:.4f}"
                 + ("  ★ BEST" if method == best_linkage_hc else ""),
                 fontsize=12, fontweight="bold",
                 color="#e94560" if method == best_linkage_hc else "black")
    ax.set_xlabel("Student groups"); ax.set_ylabel("Merge distance"); ax.grid(True, alpha=0.2, axis="y")

plt.suptitle(f"Dendrogram Comparison — All Linkage Methods  (subsample n={DENDRO_SAMPLE:,})",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

### 5.4 Optimal k Selection

In [ ]:
k_range_hc = range(2, 11)
sil_hc_list = []; dbi_hc_list = []; chi_hc_list = []

print(f"🔄 Sweeping k = 2 to 10 (linkage = {best_linkage_hc}) ...")
for k in k_range_hc:
    model_ = AgglomerativeClustering(n_clusters=k, linkage=best_linkage_hc)
    lbl_   = model_.fit_predict(X_hc_scaled)
    sil_hc_list.append(silhouette_score(X_hc_scaled, lbl_, sample_size=5000, random_state=42))
    dbi_hc_list.append(davies_bouldin_score(X_hc_scaled, lbl_))
    chi_hc_list.append(calinski_harabasz_score(X_hc_scaled, lbl_))
    print(f"   k={k:2d}  │  Sil={sil_hc_list[-1]:.4f}  │  DBI={dbi_hc_list[-1]:.4f}  │  CH={chi_hc_list[-1]:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, vals, title, best_fn in zip(
    axes,
    [sil_hc_list, dbi_hc_list, chi_hc_list],
    ["Silhouette ↑","Davies-Bouldin ↓","Calinski-Harabasz ↑"],
    [np.argmax, np.argmin, np.argmax]
):
    bk = list(k_range_hc)[best_fn(vals)]
    ax.plot(k_range_hc, vals, marker='o', linewidth=2.5, color='#2E86AB', markerfacecolor='#e94560')
    ax.axvline(bk, color='red', linestyle='--', linewidth=2, label=f'Best k={bk}')
    ax.set_title(title, fontweight='bold'); ax.legend(); ax.set_xticks(k_range_hc); ax.grid(True, alpha=0.3)
plt.suptitle(f"Optimal k — Agglomerative ({best_linkage_hc} linkage)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

best_sil_hc_k = list(k_range_hc)[np.argmax(sil_hc_list)]
best_dbi_hc_k = list(k_range_hc)[np.argmin(dbi_hc_list)]
best_chi_hc_k = list(k_range_hc)[np.argmax(chi_hc_list)]
votes_hc = Counter([best_sil_hc_k, best_dbi_hc_k, best_chi_hc_k])
OPTIMAL_K_HC = votes_hc.most_common(1)[0][0]
print(f"\n🎯 Optimal k = {OPTIMAL_K_HC}  (Sil={best_sil_hc_k}, DBI={best_dbi_hc_k}, CH={best_chi_hc_k})")

### 5.5 Train Final Agglomerative Model

In [ ]:
hc_model  = AgglomerativeClustering(n_clusters=OPTIMAL_K_HC, linkage=best_linkage_hc)
HC_LABELS = hc_model.fit_predict(X_hc_scaled)
df_hc["HC_Cluster"] = HC_LABELS

HC_SIL = silhouette_score(X_hc_scaled, HC_LABELS, sample_size=5000, random_state=42)
HC_DBI = davies_bouldin_score(X_hc_scaled, HC_LABELS)
HC_CHI = calinski_harabasz_score(X_hc_scaled, HC_LABELS)

print(f"✅ Hierarchical model trained  (linkage={best_linkage_hc}, k={OPTIMAL_K_HC})")
print(f"   Silhouette: {HC_SIL:.4f}  |  DBI: {HC_DBI:.4f}  |  CH: {HC_CHI:.2f}")
counts_hc = pd.Series(HC_LABELS).value_counts().sort_index()
for i, cnt in counts_hc.items():
    print(f"   Cluster {i}: {cnt:,} students ({cnt/len(HC_LABELS)*100:.1f}%)")

### 5.6 Cluster Profiling & Visualization

In [ ]:
# Auto-label
centroid_exam_hc = df_hc.groupby("HC_Cluster")["ExamScore"].mean()
ranked_hc = centroid_exam_hc.sort_values(ascending=False).index.tolist()
hc_label_map = {}
hc_label_map[ranked_hc[0]]  = "🟢 High Performers"
hc_label_map[ranked_hc[-1]] = "🔴 At-Risk Students"
for cl in ranked_hc[1:-1]:
    hc_label_map[cl] = "🟡 Steady Learners"
df_hc["HC_Profile"] = df_hc["HC_Cluster"].map(hc_label_map)

print("=" * 60)
print("  HIERARCHICAL CLUSTER PROFILES")
print("=" * 60)
for cl in ranked_hc:
    cnt = (df_hc["HC_Cluster"] == cl).sum()
    print(f"\n{hc_label_map[cl]}  (Cluster {cl})  |  {cnt:,} students ({cnt/len(df_hc)*100:.1f}%)")
    row = df_hc[df_hc["HC_Cluster"]==cl][CLUSTERING_FEATURES_HC].mean()
    for feat in ["ExamScore","Attendance","StudyHours","AssignmentCompletion","Motivation","StressLevel"]:
        print(f"  • {feat:<25} {row[feat]:.3f}")

# Feature heatmap
profile_hc = df_hc.groupby("HC_Cluster")[CLUSTERING_FEATURES_HC].mean().T
profile_hc.columns = [f"Cluster {c}" for c in profile_hc.columns]
p_norm_hc  = (profile_hc - profile_hc.min(axis=1).values.reshape(-1,1)) /              (profile_hc.max(axis=1) - profile_hc.min(axis=1)).values.reshape(-1,1)
fig, ax = plt.subplots(figsize=(max(8, OPTIMAL_K_HC*2+2), 8))
sns.heatmap(p_norm_hc, annot=profile_hc.round(2), fmt=".2f", cmap="RdYlGn",
            linewidths=0.5, linecolor="white", ax=ax,
            cbar_kws={"label":"Normalised mean (0–1)", "shrink":0.7})
ax.set_title("Hierarchical — Cluster Feature Heatmap", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout(); plt.show()

In [ ]:
# PCA & t-SNE
pca_hc   = PCA(n_components=2, random_state=42)
X_pca_hc = pca_hc.fit_transform(X_hc_scaled)
exp_hc   = pca_hc.explained_variance_ratio_ * 100
cmap_hc  = plt.cm.get_cmap("tab10", OPTIMAL_K_HC)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for i in range(OPTIMAL_K_HC):
    m = HC_LABELS == i
    axes[0].scatter(X_pca_hc[m,0], X_pca_hc[m,1], c=[cmap_hc(i)], alpha=0.65, s=18, label=f"Cluster {i}")
axes[0].set_xlabel(f"PC1 ({exp_hc[0]:.1f}% var)", fontweight="bold")
axes[0].set_ylabel(f"PC2 ({exp_hc[1]:.1f}% var)", fontweight="bold")
axes[0].set_title("Hierarchical — PCA Projection", fontsize=13, fontweight="bold")
axes[0].legend(markerscale=2); axes[0].grid(True, alpha=0.2)

TSNE_N_HC = min(4000, len(X_hc_scaled))
idx_t_hc  = np.random.choice(len(X_hc_scaled), TSNE_N_HC, replace=False)
print("🔄 Running t-SNE ...")
tsne_hc   = TSNE(n_components=2, perplexity=40, learning_rate="auto", init="pca", random_state=42)
X_t_hc    = tsne_hc.fit_transform(X_hc_scaled[idx_t_hc])
lbl_t_hc  = HC_LABELS[idx_t_hc]
for i in range(OPTIMAL_K_HC):
    m = lbl_t_hc == i
    axes[1].scatter(X_t_hc[m,0], X_t_hc[m,1], c=[cmap_hc(i)], alpha=0.70, s=15, label=f"Cluster {i}")
axes[1].set_xlabel("t-SNE dim 1", fontweight="bold"); axes[1].set_ylabel("t-SNE dim 2", fontweight="bold")
axes[1].set_title(f"Hierarchical — t-SNE Projection (n={TSNE_N_HC:,})", fontsize=13, fontweight="bold")
axes[1].legend(markerscale=2); axes[1].grid(True, alpha=0.2)
print("✅ t-SNE complete.")
plt.suptitle("Hierarchical Clustering — Visualization", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
hc_out_path = Path("../Data/processed/hierarchical_labelled_dataset.csv")
df_hc.to_csv(hc_out_path, index=False)
print(f"✅ Hierarchical results saved → {hc_out_path}")

---
## Phase 6 — Evaluation & Three-Algorithm Comparison

This final phase delivers a rigorous side-by-side evaluation of all three clustering approaches on identical data, interprets the student profiles, and derives actionable academic insights.

### 6.1 Reload All Models on Common Feature Set

In [ ]:
df_eval = pd.read_csv(PROCESSED_DATA_PATH)
feature_cols_eval = [c for c in df_eval.columns if c != 'FinalGrade']
X_eval = df_eval[feature_cols_eval].astype(float)
scaler_eval = StandardScaler()
X_eval_scaled = scaler_eval.fit_transform(X_eval)

# K-Means (k=4 for richer profiling in evaluation)
km_eval = KMeans(n_clusters=4, random_state=42, n_init=20)
km_eval_labels = km_eval.fit(X_eval_scaled).predict(X_eval_scaled)

# DBSCAN
dbs_eval_path = Path("../Data/processed/dbscan_labelled_dataset.csv")
if dbs_eval_path.exists():
    dbscan_eval_labels = pd.read_csv(dbs_eval_path)['DBSCAN_Cluster'].astype(int).to_numpy()
else:
    dbs_eval = DBSCAN(eps=BEST_EPS_DBS, min_samples=BEST_MS_DBS, n_jobs=-1)
    dbscan_eval_labels = dbs_eval.fit_predict(X_eval_scaled)

# Hierarchical (k=4)
hc_eval = AgglomerativeClustering(n_clusters=4, linkage='ward')
hc_eval_labels = hc_eval.fit_predict(X_eval_scaled)

print("✅ All models applied to evaluation feature set.")
print(f"   K-Means unique labels      : {np.unique(km_eval_labels)}")
print(f"   DBSCAN unique labels       : {np.unique(dbscan_eval_labels)}")
print(f"   Hierarchical unique labels : {np.unique(hc_eval_labels)}")

### 6.2 Quantitative Metrics Summary

In [ ]:
def compute_metrics(X, labels, name):
    unique = np.unique(labels)
    if len(unique) < 2:
        return None
    return {
        'Algorithm': name,
        'Clusters' : len(unique),
        'Silhouette Score ↑'     : round(silhouette_score(X, labels, sample_size=5000, random_state=42), 4),
        'Davies-Bouldin Index ↓' : round(davies_bouldin_score(X, labels), 4),
        'Calinski-Harabasz ↑'    : round(calinski_harabasz_score(X, labels), 2),
    }

metrics_all = []
for name, lbl in [("K-Means (k=4)", km_eval_labels),
                  ("DBSCAN", dbscan_eval_labels),
                  ("Hierarchical (k=4)", hc_eval_labels)]:
    r = compute_metrics(X_eval_scaled, lbl, name)
    if r: metrics_all.append(r)

metrics_eval_df = pd.DataFrame(metrics_all)
print("=" * 80)
print("  THREE-ALGORITHM EVALUATION SUMMARY")
print("=" * 80)
print(metrics_eval_df.to_string(index=False))
print("=" * 80)

### 6.3 Visual Comparison — PCA & t-SNE (All Three Algorithms)

In [ ]:
pca_eval = PCA(n_components=2, random_state=42)
pca_eval_xy = pca_eval.fit_transform(X_eval_scaled)
pca_eval_df = pd.DataFrame(pca_eval_xy, columns=['PC1','PC2'])
pca_eval_df['KMeans'] = km_eval_labels.astype(str)
pca_eval_df['DBSCAN'] = dbscan_eval_labels.astype(str)
pca_eval_df['Hierarchical'] = hc_eval_labels.astype(str)

fig, axes = plt.subplots(1, 3, figsize=(22, 6), constrained_layout=True)
pal_eval = sns.color_palette('tab10')
for ax, col, title in zip(axes,
    ['KMeans','DBSCAN','Hierarchical'],
    ['K-Means','DBSCAN','Hierarchical']):
    sns.scatterplot(data=pca_eval_df, x='PC1', y='PC2', hue=col, palette=pal_eval,
                    s=30, alpha=0.70, ax=ax, legend='brief')
    ax.set_title(f'PCA — {title}', fontsize=13, fontweight='bold')
    ax.legend(title=col, bbox_to_anchor=(1.02,1), loc='upper left')
fig.suptitle('PCA Projection of Student Clusters — All Algorithms', fontsize=16, fontweight='bold')
plt.show()

In [ ]:
# t-SNE comparison
sample_n_eval = min(4000, len(X_eval_scaled))
rng_eval = np.random.RandomState(42)
sidx = rng_eval.choice(len(X_eval_scaled), sample_n_eval, replace=False)
print("🔄 Running t-SNE for evaluation comparison ...")
tsne_eval = TSNE(n_components=2, perplexity=35, learning_rate=200, max_iter=1000, random_state=42, init='pca')
tsne_eval_xy = tsne_eval.fit_transform(X_eval_scaled[sidx])
tsne_eval_df = pd.DataFrame(tsne_eval_xy, columns=['T1','T2'])
tsne_eval_df['KMeans']       = km_eval_labels[sidx].astype(str)
tsne_eval_df['DBSCAN']       = dbscan_eval_labels[sidx].astype(str)
tsne_eval_df['Hierarchical'] = hc_eval_labels[sidx].astype(str)

fig, axes = plt.subplots(1, 3, figsize=(22, 6), constrained_layout=True)
for ax, col, title in zip(axes, ['KMeans','DBSCAN','Hierarchical'],
                           ['K-Means','DBSCAN','Hierarchical']):
    sns.scatterplot(data=tsne_eval_df, x='T1', y='T2', hue=col, palette=pal_eval,
                    s=25, alpha=0.70, ax=ax, legend='brief')
    ax.set_title(f't-SNE — {title}', fontsize=13, fontweight='bold')
    ax.legend(title=col, bbox_to_anchor=(1.02,1), loc='upper left')
fig.suptitle('t-SNE Visualization of Student Clusters — All Algorithms', fontsize=16, fontweight='bold')
plt.show()
print("✅ t-SNE complete.")

### 6.4 Cluster Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6), constrained_layout=True)
for ax, alg, lbl in zip(axes,
    ['K-Means','DBSCAN','Hierarchical'],
    [km_eval_labels, dbscan_eval_labels, hc_eval_labels]):
    counts_ev = pd.Series(lbl).value_counts().sort_index()
    sns.barplot(x=counts_ev.index.astype(str), y=counts_ev.values, palette='Set2', ax=ax)
    ax.set_title(f'{alg} — Cluster Counts', fontsize=13, fontweight='bold')
    ax.set_xlabel('Cluster'); ax.set_ylabel('Students')
    for i, v in enumerate(counts_ev.values):
        ax.text(i, v + max(counts_ev.values)*0.01, str(v), ha='center', fontsize=10)
fig.suptitle('Cluster Distribution by Algorithm', fontsize=16, fontweight='bold')
plt.show()

### 6.5 Cluster Profile Analysis & Student Labeling

In [ ]:
def describe_clusters(labels, df_ref):
    grouped = df_ref.assign(Cluster=labels).groupby('Cluster')
    summary = grouped[['StudyHours','Attendance','Resources','Extracurricular','Motivation',
                        'OnlineCourses','Discussions','AssignmentCompletion','ExamScore',
                        'StressLevel','FinalGrade']].mean()
    summary['Count'] = grouped.size()
    summary['Engagement'] = summary[['Attendance','OnlineCourses','Discussions',
                                     'Extracurricular','Resources']].mean(axis=1)
    return summary.round(3)

def assign_profile_names(summary):
    base = summary.drop(-1) if -1 in summary.index else summary
    high_perf  = base['FinalGrade'].idxmax()
    at_risk    = base['FinalGrade'].idxmin()
    remaining  = base.drop([high_perf, at_risk], errors='ignore')
    mapping = {high_perf: '🌟 High Performers', at_risk: '⚠️ At-Risk Students'}
    if len(remaining) >= 2:
        dl_score   = remaining['StressLevel'] - remaining['AssignmentCompletion']
        last_min   = dl_score.idxmax()
        passive    = remaining.drop(last_min).index[0]
        mapping[last_min] = '⏰ Last-Minute Learners'
        mapping[passive]  = '😴 Passive Students'
    elif len(remaining) == 1:
        mapping[remaining.index[0]] = '⏰ Last-Minute Learners'
    if -1 in summary.index:
        mapping[-1] = '🚩 Atypical / Noise'
    return mapping

km_sum   = describe_clusters(km_eval_labels, df_eval)
dbs_sum  = describe_clusters(dbscan_eval_labels, df_eval)
hc_sum   = describe_clusters(hc_eval_labels, df_eval)

km_map   = assign_profile_names(km_sum)
dbs_map  = assign_profile_names(dbs_sum)
hc_map   = assign_profile_names(hc_sum)

print("📌 K-Means Profile Map:", km_map)
print("📌 DBSCAN Profile Map :", dbs_map)
print("📌 Hierarchical Map   :", hc_map)

### 6.6 Behavioral Feature Heatmaps

In [ ]:
visual_feats = ['StudyHours','Attendance','AssignmentCompletion','ExamScore','Motivation']
cluster_sums = {'K-Means': km_sum[visual_feats], 'DBSCAN': dbs_sum[visual_feats], 'Hierarchical': hc_sum[visual_feats]}

fig, axes = plt.subplots(1, 3, figsize=(24, 8), constrained_layout=True)
for ax, (name, summary) in zip(axes, cluster_sums.items()):
    sns.heatmap(summary.T, annot=True, fmt='.2f', cmap='Spectral', cbar=True, ax=ax)
    ax.set_title(f'{name} — Feature Means by Cluster', fontsize=13, fontweight='bold')
fig.suptitle('Behavioral Feature Heatmaps by Cluster', fontsize=16, fontweight='bold')
plt.show()

### 6.7 Three-Algorithm Metric Comparison Chart

In [ ]:
labels_alg_eval = ['K-Means
(k=4)', 'DBSCAN', 'Hierarchical
(k=4)']
alg_sil = [metrics_eval_df.loc[0,'Silhouette Score ↑'], metrics_eval_df.loc[1,'Silhouette Score ↑'], metrics_eval_df.loc[2,'Silhouette Score ↑']]
alg_dbi = [metrics_eval_df.loc[0,'Davies-Bouldin Index ↓'], metrics_eval_df.loc[1,'Davies-Bouldin Index ↓'], metrics_eval_df.loc[2,'Davies-Bouldin Index ↓']]
alg_chi = [metrics_eval_df.loc[0,'Calinski-Harabasz ↑'], metrics_eval_df.loc[1,'Calinski-Harabasz ↑'], metrics_eval_df.loc[2,'Calinski-Harabasz ↑']]

x_eval = np.arange(3)
colors_eval = ['#2E86AB','#e94560','#06A77D']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, vals, title in zip(axes,
    [alg_sil, alg_dbi, [c/100 for c in alg_chi]],
    ['Silhouette Score ↑','Davies-Bouldin Index ↓','Calinski-Harabasz ↑ (÷100)']):
    bars_ev = ax.bar(x_eval, vals, width=0.55, color=colors_eval, alpha=0.88, edgecolor='white', linewidth=1.5)
    ax.bar_label(bars_ev, fmt='%.3f', padding=4, fontsize=11, fontweight='bold')
    ax.set_xticks(x_eval); ax.set_xticklabels(labels_alg_eval, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.25, axis='y'); ax.set_ylim(0, max(vals)*1.3)
plt.suptitle('Three-Algorithm Cluster Quality Comparison', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

### 6.8 Final Conclusions & Academic Insights

**Cluster Profile Labels:**
- 🌟 **High Performers** — high study hours, attendance, and assignment completion; strong exam scores
- ⏰ **Last-Minute Learners** — elevated stress relative to assignment completion; moderate engagement
- 😴 **Passive Students** — below-average engagement across all dimensions; moderate performance
- ⚠️ **At-Risk Students** — lowest-performing group; requires priority early intervention

**Algorithm Recommendations:**

| Algorithm | Best Used For |
|-----------|---------------|
| **K-Means** | Quick baseline segmentation; interpretable centroids for advising |
| **DBSCAN** | Outlier / atypical student detection; does not require k upfront |
| **Hierarchical** | Visual tree exploration; nested cluster structure; dendrogram interpretation |

**Key Educational Takeaways:**
1. A small but identifiable at-risk group exists across all algorithms — early outreach is justified.
2. Exam score, attendance, and assignment completion are the most discriminating features.
3. Motivation and stress level provide complementary signals for mental health and counseling referrals.
4. All three algorithms produce consistent broad segmentation, validating the clustering structure.

In [ ]:
print("=" * 80)
print("  ✅ PIPELINE COMPLETE")
print("=" * 80)
print("\nPhases completed:")
phases = [
    ("Phase 1", "Data Collection & Preprocessing"),
    ("Phase 2", "Exploratory Data Analysis"),
    ("Phase 3", "K-Means Clustering"),
    ("Phase 4", "DBSCAN Clustering"),
    ("Phase 5", "Hierarchical Clustering"),
    ("Phase 6", "Evaluation & Three-Algorithm Comparison"),
]
for code_p, desc in phases:
    print(f"  ✅ {code_p}: {desc}")
print("\nStudent profiles identified:")
for label in ['🌟 High Performers','⏰ Last-Minute Learners','😴 Passive Students','⚠️ At-Risk Students','🚩 Atypical / Noise (DBSCAN)']:
    print(f"  {label}")